# SIGK - Projekt 4: Transformacja 3D

## Cel projektu

Celem projektu było zaprojektowanie i wytrenowanie modeli sieci neuronowych realizujących **transformację geometryczną chmury punktów 3D** - przekształcenie powierzchni jednego obiektu w kształt docelowy (czajnik/teapot). Każdy z modeli trenowany był na osobnym obiekcie źródłowym: armadillo, bunny i dragon. Ewaluacja obejmuje zarówno trening na własnym kształcie, jak i generalizację na nowym, niewidzianym podczas treningu kształcie (Asian Dragon).

---

## Model Bunny → Teapot

Autor: Dominika Boguszewska


---

## Model Dragon → Teapot

Autor: Dominika Boguszewska


---

## Model Armadillo → Teapot

Autor: Filip Langiewicz



---
### Przygotowanie zbioru danych

Projekt nie korzysta z gotowego, statycznego zbioru danych - dane generowane są w locie podczas treningu 
na podstawie dwóch plików `.obj`: modelu źródłowego i docelowego.

Z siatki trójkątnej każdego modelu chmura punktów pobierana jest metodą równomiernego próbkowania powierzchni. Funkcja losuje punkty proporcjonalnie do pola każdego trójkąta - gęstość próbkowania jest jednolita niezależnie od lokalnej gęstości siatki.

Po próbkowaniu każda chmura normalizowana jest do jednostkowej sfery - wyśrodkowanie do zera (odjęcie centroidu) i skalowanie tak, by maksymalna odległość punktu od środka wynosiła 1.

Przy `augment=True` do każdej pary chmur stosowane są losowe przekształcenia (rotacja 3D i skalowanie z zakresu `(0.75, 1.25)`), identycznie dla chmury źródłowej i docelowej - model uczy się transformacji niezależnej od orientacji i skali sceny.


| Zbiór       | Liczba próbek |
|-------------|---------------|
| Treningowy  | 9 000         |
| Walidacyjny | 1 000         |


---
### Architektura modelu - VectorFieldNet

Model rozwiązuje zadanie jako **regresję pola przesunięć** (*displacement field regression*): 
zamiast przewidywać bezwzględne współrzędne punktów docelowych, dla każdego punktu wejściowego `x_i` 
model przewiduje wektor przesunięcia `Δx_i`, a punkt wyjściowy wyznaczany jest jako `x_i + Δx_i`. 
Takie podejście sprawia, że sieć uczy się jedynie *różnicy kształtów*, nie odtwarzając kształtu od zera - co znacznie stabilizuje uczenie.

### Wejście

Tensor `(B, N, 3)` - batch `B` chmur po `N=2048` punktów, każdy opisany współrzędnymi XYZ.

### Blok 1 - Lokalny enkoder punktów

Chmura spłaszczana jest do (B·N, 3) i każdy punkt przetwarzany jest niezależnie przez wspólną MLP:

```
Linear(3 → 64) → BatchNorm1d(64) → ReLU
Linear(64 → 128) → BatchNorm1d(128) → ReLU
```

Spłaszczenie do (B·N, 3) jest konieczne, bo BatchNorm1d wymaga tensora 2D. Wagi są współdzielone między wszystkimi punktami, więc sieć uczy się lokalnych deskryptorów geometrycznych niezależnie od permutacji zbioru. Wynik: (B·N, 128), reshapowany do (B, N, 128).

### Blok 2 - Globalny deskryptor kształtu

Z lokalnych cech (B, N, 128) wyznaczany jest globalny kontekst przez max-pooling wzdłuż osi punktów: (B, N, 128) → (B, 128). Max-pooling wybiera najbardziej aktywną cechę spośród wszystkich punktów. Globalny wektor (B, 128) przetwarzany jest przez kolejną MLP:

```
Linear(128 → 256) → BatchNorm1d(256) → ReLU
Linear(256 → 512) → BatchNorm1d(512) → ReLU
```

Wynik (B, 512) jest rozszerzany (expand) i konkatenowany z lokalnymi cechami każdego punktu, dając (B·N, 128+512) = (B·N, 640). Każdy punkt „widzi" więc zarówno swój lokalny kontekst geometryczny, jak i informację o strukturze całego kształtu.

### Blok 3 - Dekoder

Połączone cechy (B·N, 640) przetwarzane są przez MLP:

```
Linear(640 → 256) → BatchNorm1d(256) → ReLU
Linear(256 → 128) → BatchNorm1d(128) → ReLU
Linear(128 → 3)   — bez aktywacji
```

Wynik (B·N, 3) reshapowany do (B, N, 3) to wektory przesunięć, dodawane do punktów wejściowych: x_pred = x_input + Δx.

### Wyjście

Tensor `(B, N, 3)` - predykowana chmura punktów w kształcie czajnika.

| Parametr             | Wartość     |
|----------------------|-------------|
| `local_hidden_dims`  | [64, 128]   |
| `global_hidden_dims` | [256, 512]  |
| `output_hidden_dims` | [256, 128]  |
| Liczba parametrów    | **373 251** |


---
### Funkcja straty - Chamfer Distance

Jako funkcję straty zastosowano **Chamfer Distance (CD)**, która mierzy geometryczne podobieństwo między dwiema 
chmurami punktów bez wymaganej korespondencji punktów.

$$\mathcal{L}_{CD}(\hat{P}, Q) = \frac{1}{|\hat{P}|}\sum_{p \in \hat{P}} \min_{q \in Q} \|p - q\|^2 + \frac{1}{|Q|}\sum_{q \in Q} \min_{p \in \hat{P}} \|q - p\|^2$$

Pierwszy składnik karze predykowane punkty bez bliskiego odpowiednika w zbiorze docelowym. 
Drugi odwrotnie - karze za punkty docelowe nieodwzorowane przez predykcję. 

---

### Trening - dwuetapowy

| Etap | Epoki | LR   | Val Loss start | Val Loss koniec |
|------|-------|------|----------------|-----------------|
| 1    | 100   | 1e-3 | 0.003517       | 0.001279        |
| 2    | 200   | 3e-4 | 0.001313       | **0.001164**    |

Po pierwszym etapie treningu zdecydowano się kontynuować trening na kolejnych 200 epokach. Straty podczas obu etapów zaprezentowane są na poniższych wykresach.

![ArmadilloTrain1](img/loss_curve1.png)
![ArmadilloTrain1](img/loss_curve2.png)

---
### Rezultat Armadillo -> Teapot

In [10]:
from armadillo.dataset import load_obj_pointcloud, normalize_pointcloud
from armadillo.utils import visualize_transition_3d, interpolate_pointclouds
from pathlib import Path
import numpy as np

RESULTS_DIR    = Path("results")
ASIAN_DRAG_PATH = Path("modele/asian_dragon_really_small.obj")
N_POINTS = 2048


def show_transition_from_npy(src_path: Path, pred_npy: Path, src_title: str) -> np.ndarray:
    src_pts  = normalize_pointcloud(load_obj_pointcloud(str(src_path), N_POINTS))
    pred_pts = np.load(pred_npy)
    steps    = interpolate_pointclouds(src_pts, pred_pts, steps=4)
    visualize_transition_3d(
        steps,
        titles=[src_title, "Step 1", "Step 2", "Teapot (predicted)"]
    )
    return pred_pts

In [11]:
print("=== Armadillo → Teapot ===")
ARMADILLO_PATH = Path("modele/armadillo_small.obj")
pred_armadillo = show_transition_from_npy(
    ARMADILLO_PATH,
    RESULTS_DIR / "pred_armadillo.npy",
    "Armadillo"
)

=== Armadillo → Teapot ===


---
### Rezultaty Asian Dragon -> Teapot

In [13]:
print("=== Asian Dragon → Teapot (all three flows) ===")
# pred_asian_bunny  = show_transition_from_npy(
#     ASIAN_DRAG_PATH,
#     RESULTS_DIR / "pred_asian_bunny.npy",
#     "Asian Dragon (bunny flow)"
# )
# pred_asian_dragon = show_transition_from_npy(
#     ASIAN_DRAG_PATH,
#     RESULTS_DIR / "pred_asian_dragon.npy",
#     "Asian Dragon (dragon flow)"
# )
pred_asian_armadillo = show_transition_from_npy(
    ASIAN_DRAG_PATH,
    RESULTS_DIR / "pred_asian_armadillo.npy",
    "Asian Dragon (armadillo flow)"
)

=== Asian Dragon → Teapot (all three flows) ===


---

## Wyniki

| Przepływ                                  | IoU↑   | Dice↑  | Chamfer↓ |
|-------------------------------------------|--------|--------|----------|
| bunny_flow                                | -      | -      | -        |
| dragon_flow                               | -      | -      | -        |
| armadillo_flow                            | 0.7484 | 0.8561 | 3.2653   |
| bunny_flow_asian_dragon                   | -      | -      | -        |
| dragon_flow_asian_dragon                  | -      | -      | -        |
| armadillo_flow_asian_dragon               | 0.7942 | 0.8853 | 3.1840   |




---
### Wnioski

Model `VectorFieldNet` skutecznie realizuje geometryczną transformację chmury punktów 3D. 
Architektura ta pozwala na dobre uogólnienie - model wytrenowany na armadillo poprawnie transformuje niewidziany Asian Dragon, uzyskując nawet nieznacznie lepsze metryki niż dla obiektu treningowego.

Dwuetapowy trening z dotrenowaniem od checkpointu przy niższym LR okazał się skuteczną strategią - pozwolił na dodatkowe ~9% poprawy funkcji straty na zbiorze walidacyjnym. Metryki dla asian dragon uległy poprawie, zaś dla armadillo - nieznacznie się pogorszyły.